In [1]:
# =============================================================================
# CELL 1: ALL CONFIGURATION, ASSUMPTIONS, BASELINES, INPUTS
# =============================================================================

# --- Granularity: 'q' = quarterly, 'm' = monthly, 'w' = weekly ---
granularity = 'w'

# --- Date Range (inclusive) ---
START_DATE = '2025-10-07'
END_DATE = None  # None = auto-detect from today's date

# --- Query Control ---
run_every_query = True  # True = run SQL queries; False = use cached pickles

# --- Date Column per Granularity ---
# Q/M use book_date; W uses app_date (application_received_dtm)
DATE_COL_MAP = {'q': 'book_date', 'm': 'book_date', 'w': 'app_date'}

# --- LOBs to Process ---
LOBS = ['STE']  # Distinct from STG in bareboned_ragu_new.ipynb

# --- Rollup Groups ---
ROLLUP_GROUPS = {}  # No rollups needed (single LOB); control room adds STE to POS/nonKMX later

# --- Baselines (STE-specific) ---
BASELINES = {
    'STE': {'ltv': 1.94, 'new_recovery_unadjusted': 0.557, 'apr': 0.25},
}

# --- Model Parameters ---
MODEL_PARAMS = {
    'mmi_standard_increase': 1.03,
    'expected_years_on_book': 2,
    'impound_probability': 0.15,
    'mean_unit_loss': 0.5,
    'unit_loss_to_model_score': 0.02,
    'skip_rate': 0.75,
    'kmx_loss_scale': 0.067,
}

# --- STE does NOT use DLA -- uniform pricing scalar ---
PRICING_SCALAR = 1.0

# --- Dual-path override: MANDATORY for STE (new LOB with outlier exposure) ---
USE_STE_METRICS = True  # Override MS/LTV/APR from ste_ragu_weekly.txt

# --- Excluded Vintages ---
EXCLUDED_VINTAGES = {}

In [2]:
# Parameters
granularity = "w"
START_DATE = "2025-10-07"
END_DATE = None
run_every_query = True
BASELINES = {"STE": {"ltv": 1.94, "new_recovery_unadjusted": 0.557, "apr": 0.25}}
MODEL_PARAMS = {"mmi_standard_increase": 1.03, "expected_years_on_book": 2, "impound_probability": 0.15, "mean_unit_loss": 0.5, "unit_loss_to_model_score": 0.02, "skip_rate": 0.75, "kmx_loss_scale": 0.067}
run_sandbox = False
run_historical = False


In [3]:
# =============================================================================
# CELL 2: IMPORTS AND DERIVED CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import time
import datetime as dt
import re
import os
import openpyxl
from tqdm.notebook import tqdm

tqdm.pandas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)

# --- Derived values (do not modify) ---
date_col = DATE_COL_MAP[granularity]
start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp.today().normalize()

PERIOD_FREQ_MAP = {'q': 'Q', 'm': 'M', 'w': 'W-SAT'}
period_freq = PERIOD_FREQ_MAP[granularity]

start_period = start_date.to_period(period_freq)
end_period = end_date.to_period(period_freq)

min_date_sql = f"'{START_DATE}'"

print(f"Granularity: {granularity}")
print(f"Date column: {date_col}")
print(f"Period range: {start_period} to {end_period}")
print(f"SQL min_date: {min_date_sql}")

Granularity: w
Date column: app_date
Period range: 2025-10-05/2025-10-11 to 2026-05-03/2026-05-09
SQL min_date: '2025-10-07'


In [4]:
# =============================================================================
# CELL 3: UTILITY FUNCTIONS
# =============================================================================

def run_sql(filename, sub_list=None, connection=None, filename_is_query=False):
    if sub_list is None:
        sub_list = []
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for text, var in sub_list:
        query = query.replace(text, var)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df


def store_pickle(data, filename):
    if isinstance(data, str):
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


def cached_sql(filename, pickle_name, sub_list=None, connection=None, force_refresh=False):
    """Fetch from SQL and cache to pickle. Reuse cache unless force_refresh=True
    or the pickle file is missing."""
    if force_refresh or not os.path.exists(pickle_name):
        df = run_sql(filename, sub_list=sub_list, connection=connection)
        store_pickle(df, pickle_name)
        return df
    return get_pickle(pickle_name)


def smooth(series):
    averaged_series = pd.Series(index=series.index, dtype=float)
    averaged_series.iloc[0] = series.iloc[0]
    averaged_series.iloc[1] = (series.iloc[0] + series.iloc[1] + series.iloc[2]) / 3
    for i in range(2, len(series) - 2):
        averaged_series.iloc[i] = series.iloc[i-2:i+3].mean()
    averaged_series.iloc[-2] = (series.iloc[-1] + series.iloc[-2] + series.iloc[-3]) / 3
    averaged_series.iloc[-1] = series.iloc[-1]
    return averaged_series


def weight_by_proceeds(metric, proceeds):
    return (metric * proceeds).sum() / proceeds.sum()


def weighted_average_and_sum(group, metrics):
    if isinstance(metrics, str):
        weighted_avg = (group[metrics] * group.amt_financed).sum() / group.amt_financed.sum()
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        weighted_avg = (group[metric] * group.amt_financed).sum() / group.amt_financed.sum()
        result_dict[metric] = weighted_avg
    return pd.Series(result_dict)


def assign_period(df, col_name, freq):
    """Assign a pd.Period column from a date column."""
    dt_series = pd.to_datetime(df[col_name])
    return dt_series.dt.to_period(freq)


def format_vintage(period_series):
    """Convert pd.Period series to formatted vintage strings (e.g. '2025 Q1', '2025 M01')."""
    freq = period_series.iloc[0].freqstr
    if freq == 'Q-DEC':
        return period_series.dt.year.astype(str) + ' Q' + period_series.dt.quarter.astype(str)
    elif freq == 'M':
        return period_series.dt.year.astype(str) + ' M' + period_series.dt.month.astype(str).str.zfill(2)
    return period_series.astype(str)


def _ste_weighted_avg(g, metric_col, weight_col='con_amount_financed_back'):
    """Amount-financed-weighted average, ignoring NaN in metric."""
    mask = g[metric_col].notna()
    if not mask.any():
        return np.nan
    return (g.loc[mask, metric_col] * g.loc[mask, weight_col]).sum() / g.loc[mask, weight_col].sum()

In [5]:
# =============================================================================
# CELL 4: ULA MULTIPLIER FUNCTIONS
# =============================================================================

def get_ula_multiplier_nonkmx(ula_df, leave_out='None'):
    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag

    if leave_out != 'Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag

    if leave_out != 'High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Car make':
        ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                                   - 0.1 * ula_df.car_make_benefit_flag
                                   - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)

    if leave_out != 'Theft risk':
        ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                                   + 0.013 * ula_df.pricing_change_flag)

    if leave_out != 'MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag

    if leave_out != 'Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag

    if leave_out != 'Student Loans':
        ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
            - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
            * (1 - ula_df.student_loan_flag)) * ula_df.student_loans_cutoff_date

    if leave_out != 'Low PTI':
        ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag

    if leave_out != 'Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag

    if leave_out != 'Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag

    if leave_out != 'Fraud':
        ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)

    if leave_out != 'Driver flag':
        ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag

    if leave_out != 'Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)

    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier *= ula_df.pricing_scalar
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)

    return ula_df


def get_ula_multiplier_kmx(ula_df, loss_scale=None, leave_out='None'):
    if loss_scale is None:
        loss_scale = MODEL_PARAMS['kmx_loss_scale']

    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Low FICO':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.25 * (ula_df.low_fico_flag & ~ula_df.high_model_score_flag)
            + loss_scale * (ula_df.low_fico_flag & ula_df.high_model_score_flag))

    if leave_out != 'Low Vantage':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.25 * (ula_df.low_vantage_flag & ~ula_df.high_model_score_flag)
            + loss_scale * (ula_df.low_vantage_flag & ula_df.high_model_score_flag))

    if leave_out != 'High PTI':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + 0.05 * ula_df.high_pti_tier_1_flag
            + 0.1 * ula_df.high_pti_tier_2_flag
            + (1.4 * (1 + loss_scale) - 1) * ula_df.high_pti_tier_3_flag)

    ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] /= (1 + loss_scale)

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + (-0.01 + 0.06 * ula_df.secured_credit_flag)
            * ~(~ula_df.job_time_flag & ula_df.narrowed_soft_pull_flag & ula_df.secured_credit_flag))

    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.99 + 0.18 * ula_df.kmx_auth_tradelines_flag

    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= (
            1 + ~ula_df.job_time_flag * (~ula_df.narrowed_soft_pull_flag * -0.025
            + ula_df.narrowed_soft_pull_flag * (~ula_df.secured_credit_flag * 0.108
            + ula_df.secured_credit_flag * 0.295)))

    if leave_out != 'Fraud':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + (ula_df.fraud_adjustment - 1)

    if leave_out != 'Clip':
        mask = (ula_df.mtn_model.isin([3.0])) & (~ula_df.high_pti_tier_3_flag)
        ula_df.loc[mask, 'loss_multiplier'] = np.clip(ula_df.loc[mask, 'loss_multiplier'], 0.8, 1.35 / (1 + loss_scale))

    if leave_out != 'Vehicle Age':
        ula_df.loc[ula_df.mtn_model.isin([3.0]), 'loss_multiplier'] *= 0.8593 + 0.0201 * ula_df.continuous_vehicle_age

    if leave_out != 'npc':
        ula_df.loc[ula_df.kmx_npc_flag, 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.high_pti_npc

    if leave_out != 'Student Loans':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.96 + 0.14 * ula_df.student_loan_flag

    if leave_out != 'high sales price':
        ula_df.loc[ula_df.mtn_model.isin([4.1]), 'loss_multiplier'] *= 0.98 + 0.22 * ula_df.high_sales_price_flag

    if leave_out != 'Driver flag':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.15 * ula_df.driver_flag

    if leave_out != 'Louisiana':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.35 * ula_df.louisiana_flag

    if leave_out != 'georgia':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.georgia_flag

    if leave_out != 'txca':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score > 140), 'loss_multiplier'] *= 1 - 0.1 * ula_df.txca_flag
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score <= 140) & (ula_df.cd_model_score >= 135), 'loss_multiplier'] *= 1 - 0.05 * ula_df.txca_flag

    if leave_out != 'state_counter_adj':
        mask = ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]) & (ula_df.cd_model_score < 150) & ~(ula_df.louisiana_flag | ula_df.georgia_flag | ula_df.txca_flag)
        ula_df.loc[mask, 'loss_multiplier'] *= 1.012

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= (
            0.96 + (0.01 * ula_df.soft_pull_flag - 0.18 * ula_df.chime_flag * ula_df.soft_pull_flag) + 0.46 * ula_df.chime_flag)

    if leave_out != 'Job time':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.21 * ula_df.job_time_flag

    if leave_out != 'Existing DQ':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.existing_dq_flag

    if leave_out != 'Employment type':
        ula_df.loc[ula_df.mtn_model.isin([3.0, 3.1, 3.2, 4.1]), 'loss_multiplier'] *= 1 + 0.1 * ula_df.seasonal_employment_flag

    if leave_out != 'Authorized tradelines':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] *= 0.99 + 0.06 * ula_df.kmx_auth_tradelines_flag

    if leave_out != 'Soft pull':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.1 * (0.99 + 0.11 * ula_df.low_bureau_flag) * (0.978 + 0.172 * ula_df.open_tl_flag)
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= (1 * (0.98 + 0.22 * ula_df.low_bureau_flag) * (0.945 + 0.405 * ula_df.open_tl_flag) / np.maximum(ula_df.cd_perc_flag * ula_df.open_tl_flag * 1.2, 1))

        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.97 + 0.15 * ula_df.cd_perc_flag
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2]) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 0.954 + 0.346 * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1) & ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.05 * ula_df.cd_perc_flag
        ula_df.loc[(ula_df.mtn_model == 4.1) & ~ula_df.soft_pull_flag, 'loss_multiplier'] *= 1.0 + 0.10 * ula_df.cd_perc_flag

    if leave_out != 'blanket adjustment':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] /= 1.1

    if leave_out != 'Clip':
        ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'] = np.clip(
            ula_df.loc[ula_df.mtn_model.isin([3.1, 3.2, 4.1]), 'loss_multiplier'], 0.75, 1.4)

    return ula_df

In [6]:
# =============================================================================
# CELL 5: DATA FETCH (SQL + PICKLE)
# =============================================================================

os.makedirs('cache', exist_ok=True)
force = run_every_query

need_conn = force or not all(
    os.path.exists(p) for p in ('cache/ste_ula_v1.pkl', 'cache/ste_recovery_v1.pkl', 'cache/ste_weekly_v1.pkl')
)

if need_conn:
    conn = pyodbc.connect("DSN=Redshift_prod_new")

    # Temp tables must be created first on the same connection
    with open('ste_ragu_temptables.txt', 'r') as f:
        conn.execute(f.read().strip())
    print('Temp tables created')

    ula_df_total = cached_sql('ste_ragu_ula.txt', 'cache/ste_ula_v1.pkl',
                              connection=conn, force_refresh=force)
    print(f'ULA ready: {len(ula_df_total):,} records')

    new_recovery = cached_sql('ste_ragu_recovery.txt', 'cache/ste_recovery_v1.pkl',
                              connection=conn, force_refresh=force)
    print(f'Recovery ready: {len(new_recovery):,} records')

    ste_weekly_raw = cached_sql('ste_ragu_weekly.txt', 'cache/ste_weekly_v1.pkl',
                                connection=conn, force_refresh=force)
    print(f'STE weekly metrics ready: {len(ste_weekly_raw):,} records')

    conn.close()
else:
    ula_df_total = get_pickle('cache/ste_ula_v1.pkl')
    new_recovery = get_pickle('cache/ste_recovery_v1.pkl')
    ste_weekly_raw = get_pickle('cache/ste_weekly_v1.pkl')
    print('All data loaded from cache')

print(f"ULA records: {len(ula_df_total):,}")
print("[PROGRESS] Data Fetch Complete")


Temp tables created


ULA ready: 64,578 records


Recovery ready: 14,706 records


STE weekly metrics ready: 14,802 records


ULA records: 64,578
[PROGRESS] Data Fetch Complete


In [7]:
# =============================================================================
# CELL 6: PERIOD ASSIGNMENT + DATE FILTERING
# =============================================================================

# Filter out Core LOB if present
ula_df_total = ula_df_total[ula_df_total.lob != 'Core']

# Relabel lob from 'STG' to 'STE' (SQL returns STG; keep distinct from bareboned's STG)
ula_df_total['lob'] = 'STE'

# Ensure date columns are proper types
ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
ula_df_total['book_date'] = pd.to_datetime(ula_df_total['book_date'])
new_recovery['app_date'] = pd.to_datetime(new_recovery['app_date'])
new_recovery['book_date'] = pd.to_datetime(new_recovery['book_date'])

# Assign period columns
for df in [ula_df_total, new_recovery]:
    df['quarter'] = pd.to_datetime(df[date_col]).dt.to_period('Q')
    df['month'] = pd.to_datetime(df[date_col]).dt.to_period('M')
    df['week'] = pd.to_datetime(df[date_col]).dt.to_period('W-SAT')

    period_key = {'q': 'quarter', 'm': 'month', 'w': 'week'}
    df['period'] = df[period_key[granularity]]

# Filter to configured date range
for df in [ula_df_total, new_recovery]:
    mask = (df['period'] >= start_period) & (df['period'] <= end_period)
    df.drop(df[~mask].index, inplace=True)

# Convert week columns to str
ula_df_total['book_week'] = ula_df_total['book_week'].astype(str)

# String version of date_col for flag comparisons
ula_df_total[f'{date_col}_str'] = ula_df_total[date_col].astype(str)

# STE-specific caps
ula_df_total = ula_df_total[ula_df_total.amt_financed <= 75000]
ula_df_total = ula_df_total[ula_df_total.pti <= 0.6]
ula_df_total = ula_df_total[ula_df_total.total_income <= 200000]

print(f"Periods in data: {ula_df_total['period'].nunique()}")
print(f"Period range: {ula_df_total['period'].min()} to {ula_df_total['period'].max()}")
print(f"ULA after caps: {len(ula_df_total):,}")

Periods in data: 31


Period range: 2025-10-05/2025-10-11 to 2026-05-03/2026-05-09
ULA after caps: 62,226


In [8]:
# =============================================================================
# CELL 7: FLAG CREATION, DATA REFINEMENT, ms_df, AND DUAL-PATH METRICS
# =============================================================================

date_col_str = f'{date_col}_str'

# --- ULA Processing ---
ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model == 'MTN3.1'
ula_df_total['vehicle_age'] = np.maximum(ula_df_total[date_col_str].str[:4].astype(int) - ula_df_total.model_year, 1/365)
ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
ula_df_total.make = ula_df_total.make.str.upper().str[:3]
ula_df_total.lob_or_bucket = np.select(
    [ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
     ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])],
    ['D', 'C'], default=ula_df_total.lob_or_bucket)
ula_df_total['continuous_vehicle_age'] = (ula_df_total[date_col_str].str[:4].astype(int)
    + ula_df_total[date_col_str].str[5:7].astype(int) / 12
    - (ula_df_total.model_year - 0.25) - 1)

# --- No DLA Merge: STE uses uniform pricing scalar ---
ula_df_total['pricing_scalar'] = PRICING_SCALAR

# --- Driver Flag ---
warnings.filterwarnings("ignore", category=UserWarning)
ula_df_total.job_company = ula_df_total.job_company.fillna('')
ula_df_total['driver_flag'] = np.where(
    ula_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
    1, 0)
warnings.filterwarnings("default", category=UserWarning)

# --- NA Handling ---
ula_df_total = ula_df_total.dropna(subset=['lob'])
ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)

# --- NonKMX Flags (STE uses non_kmxent path with PTI threshold = 0.2) ---
ula_df_total['ent_fld_flag'] = False
ula_df_total['small_amt_financed_flag'] = (ula_df_total.bbvalue > 0) & (ula_df_total.bbvalue < 5000) & (ula_df_total.amt_financed < 4500)
ula_df_total['zero_cash_down_flag'] = (ula_df_total.cash_down <= 250) & (ula_df_total.tradein_value < 3000)
ula_df_total['high_mileage_vehicle_flag'] = (ula_df_total.mileage >= 100000) & (ula_df_total.cd_model_score < 130)
ula_df_total['high_pti_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_penalty_flag'] = ula_df_total.make.isin({'CAD', 'CHR', 'BMW', 'BUI', 'SUB'}) & (ula_df_total.cd_model_score < 130) & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_benefit_flag'] = (ula_df_total.cd_model_score >= 133) & ula_df_total.make.isin({'HON', 'TOY', 'LEX'}) & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['theft_risk_flag'] = ula_df_total.make.isin({'KIA', 'HYU'}) & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.model_year >= 2015) & (ula_df_total.model_year <= 2021) & (ula_df_total[date_col_str] >= '2022-07-01') & (ula_df_total[date_col_str] < '2025-01-01')
ula_df_total['mcy_low_mileage_flag'] = False
ula_df_total['weekend_flag'] = ula_df_total.day_of_week.isin([0, 6])
ula_df_total['weekday_flag'] = ula_df_total.day_of_week.isin(range(1, 6))
ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
ula_df_total['chime_flag'] = ula_df_total.chime_indicator
ula_df_total['nonkmx_chime_flag'] = ula_df_total.nonkmx_chime_indicator
ula_df_total['seasonal_employment_flag'] = ula_df_total['seasonal_employment_flag'] == 1 if 'seasonal_employment_flag' in ula_df_total.columns else (ula_df_total.get('employment', '') == 'seasonal')
ula_df_total['waiter_employment_flag'] = ula_df_total['waiter_employment_flag'] == 1 if 'waiter_employment_flag' in ula_df_total.columns else (ula_df_total.get('employment', '') == 'waiter')
ula_df_total['nonkmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines > 0.2
ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0
ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
ula_df_total['pricing_change_flag'] = ula_df_total[date_col_str] >= '2024-10-01'
ula_df_total['ent_flag'] = False
ula_df_total['student_loans_cutoff_date'] = ula_df_total[date_col_str] >= '2023-05-01'
ula_df_total['low_pti_flag'] = (ula_df_total.pti <= 0.05) & (ula_df_total.cd_model_score >= 130) & (ula_df_total.cb_flag)
ula_df_total['student_loan_flag'] = 0  # Inactive for STE

# --- KMX Flags (retained for structural parity) ---
ula_df_total['high_sales_price_flag'] = ula_df_total.sale_price > 30000
ula_df_total['kmx_npc_flag'] = False
ula_df_total['high_pti_npc'] = ula_df_total.pti > 0.2
ula_df_total['job_time_flag'] = ula_df_total.employed_months < 6
ula_df_total['low_fico_flag'] = (ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 450)
ula_df_total['high_model_score_flag'] = ula_df_total.cd_model_score >= 146
ula_df_total['low_vantage_flag'] = (ula_df_total.fico_score < 300) & (ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450)
ula_df_total['louisiana_flag'] = ula_df_total.state == 'LA'
ula_df_total['txca_flag'] = ula_df_total.state.isin(['TX', 'CA'])
ula_df_total['georgia_flag'] = ula_df_total.state == 'GA'
ula_df_total['normal_pti_flag'] = ula_df_total.pti <= 0.2
ula_df_total['high_pti_tier_1_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.pti <= 0.25)
ula_df_total['high_pti_tier_2_flag'] = (ula_df_total.pti > 0.25) & (ula_df_total.pti <= 0.35)
ula_df_total['high_pti_tier_3_flag'] = ula_df_total.pti > 0.35
ula_df_total['existing_dq_flag'] = ula_df_total.existing_dq_count > 0
ula_df_total['kmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines >= 0.14
ula_df_total['low_bureau_flag'] = ((ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 475)) | ((ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450))
ula_df_total['soft_pull_flag'] = ula_df_total.pull_type.isin(['softpull', 'prequal'])
ula_df_total['cd_perc_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1)
ula_df_total['open_tl_flag'] = ula_df_total.open_tl == 0
ula_df_total['narrowed_soft_pull_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1) & ula_df_total.soft_pull_flag & (~ula_df_total.job_time_flag)

# --- Deduplicate driver flags ---
combined_driver_flag_df = ula_df_total.groupby('account_number').driver_flag.max().reset_index()
ula_df_total = ula_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
ula_df_total = ula_df_total.drop(columns='job_company').drop_duplicates()

# --- Vintage strings ---
ula_df_total['vintage'] = format_vintage(ula_df_total['period'])
new_recovery['vintage'] = format_vintage(new_recovery['period'])

# --- Aggregate model scores from ULA data ---
ms_df = ula_df_total.groupby(['period', 'lob']).apply(
    weighted_average_and_sum, 'cd_model_score', include_groups=False
).reset_index()
ms_df = ms_df.rename(columns={'cd_model_score': 'model_score'})
ms_df['period'] = format_vintage(ms_df['period'])

print(f"ULA after refinement: {len(ula_df_total):,}")
print(f"Model scores aggregated: {len(ms_df)} period-LOB combinations")

# --- DUAL-PATH: Build ste_metrics_df from ste_ragu_weekly.txt ---
# This provides outlier-resistant vintage-level MS, LTV, APR from the SFS contract chain
ste_filtered = ste_weekly_raw.copy()
ste_filtered['book_date'] = pd.to_datetime(ste_filtered['book_date'])
ste_filtered['period'] = pd.to_datetime(ste_filtered['book_date']).dt.to_period(period_freq)
ste_filtered = ste_filtered[(ste_filtered.period >= start_period) & (ste_filtered.period <= end_period)]
ste_filtered['vintage'] = format_vintage(ste_filtered['period'])

# Apply same STE caps
ste_filtered = ste_filtered[ste_filtered['con_amount_financed_back'] <= 75000]
ste_filtered = ste_filtered[ste_filtered['con_pti_back'] <= 0.6]
ste_filtered = ste_filtered[ste_filtered['total_income'] <= 200000]
ste_filtered['bbltv'] = ste_filtered['con_amount_financed_back'] / ste_filtered['bb_value'].replace(0, np.nan)

def _build_ste_metrics(g):
    w = g['con_amount_financed_back']
    ms_valid = g['con_risk_model_score'].notnull()
    ltv_valid = g['bbltv'].notnull()
    return pd.Series({
        'model_score_wtd': (g.loc[ms_valid, 'con_risk_model_score'] * w[ms_valid]).sum() / w[ms_valid].sum() if ms_valid.any() else np.nan,
        'ltv_wtd': (g.loc[ltv_valid, 'bbltv'] * w[ltv_valid]).sum() / w[ltv_valid].sum() if ltv_valid.any() else np.nan,
        'apr_wtd': (g['con_apr'] * w).sum() / w.sum(),
        'amt_financed_total': w.sum(),
        'n_accounts': len(g),
    })

ste_metrics_df = ste_filtered.groupby('vintage').apply(_build_ste_metrics).reset_index()
print(f"\nste_metrics_df built: {len(ste_metrics_df)} vintages")
print(ste_metrics_df.head())

ULA after refinement: 14,254
Model scores aggregated: 31 period-LOB combinations

ste_metrics_df built: 31 vintages
                 vintage  model_score_wtd   ltv_wtd   apr_wtd  \
0  2025-10-05/2025-10-11       132.387991  1.382778  0.250938   
1  2025-10-12/2025-10-18       131.324983  1.542568  0.239909   
2  2025-10-19/2025-10-25       130.586113  1.508101  0.238982   
3  2025-10-26/2025-11-01       130.742756  1.540125  0.235945   
4  2025-11-02/2025-11-08       131.189521  1.524938  0.234837   

   amt_financed_total  n_accounts  
0           715101.38        22.0  
1          4811119.39       149.0  
2         12455661.46       384.0  
3         13458388.56       414.0  
4         13935056.35       455.0  


In [9]:
# =============================================================================
# CELL 8: RAGU SCORE COMPUTATION
# =============================================================================

mean_unit_loss = MODEL_PARAMS['mean_unit_loss']
unit_loss_to_model_score = MODEL_PARAMS['unit_loss_to_model_score']


def get_ragu_score(vintage, lob, ula_df_total, new_recovery, ms_df, baseline_config, leave_out='None', ste_metrics_df=None):
    """Core RAGU Score calculation for a single vintage and LOB.
    When ste_metrics_df is provided, ms_original/ltv/apr are overridden with
    SFS-sourced values (mandatory dual-path for STE)."""
    baseline_ltv = baseline_config['ltv']
    new_baseline_recovery_unadjusted_pct = baseline_config['new_recovery_unadjusted']
    baseline_apr = baseline_config['apr']

    ula_df = ula_df_total[(ula_df_total.vintage == vintage) & (ula_df_total.lob == lob)].copy()

    if len(ula_df) == 0:
        return None

    ula_df = get_ula_multiplier_nonkmx(ula_df, leave_out)

    ula_df = ula_df[['account_number', date_col, 'bbvalue', 'sale_price', 'amt_financed', 'lob_or_bucket', 'lob', 'loss_multiplier', 'apr']]

    nr = new_recovery[['account_number', 'new_recovery_multiplier']].drop_duplicates(subset='account_number', keep='first')
    mix_df = ula_df.merge(nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
                          on='account_number', how='left').drop_duplicates(subset='account_number', keep='first')

    mix_df['ltv'] = mix_df.amt_financed / mix_df.bbvalue

    bb_populated_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0)]
    if len(bb_populated_df) == 0:
        return None

    full_pop_metrics = bb_populated_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['loss_multiplier', 'ltv', 'bbvalue', 'apr'],
        include_groups=False
    )

    recovery_df = mix_df[mix_df['bbvalue'].notna() & (mix_df['bbvalue'] > 0) & mix_df['recovery_multiplier'].notna()]
    recovery_df = recovery_df.copy()
    recovery_df['recovery_unadjusted_multiplier'] = recovery_df['recovery_multiplier']
    recovery_metrics = recovery_df.groupby('lob').apply(
        weighted_average_and_sum,
        ['recovery_unadjusted_multiplier'],
        include_groups=False
    )
    recovery_metrics = recovery_metrics.drop(columns='amt_financed')

    grouped_mix_df = full_pop_metrics.join(recovery_metrics)

    vintage_ms_df = ms_df[ms_df['period'] == vintage].copy()

    index_name = grouped_mix_df.index.name
    if isinstance(index_name, str) and index_name in grouped_mix_df.columns:
        grouped_mix_df = grouped_mix_df.reset_index(drop=True)
    else:
        grouped_mix_df = grouped_mix_df.reset_index()

    full_df = grouped_mix_df.merge(vintage_ms_df, on='lob')

    if len(full_df) == 0:
        return None

    full_df['est_unit_loss'] = mean_unit_loss
    full_df['unit_loss_score'] = full_df.model_score + (1 - full_df.loss_multiplier) * full_df.est_unit_loss / unit_loss_to_model_score
    full_df = full_df.set_index('lob')

    full_df['ms_original'] = full_df.model_score.copy()
    full_df['baselined_unadjusted_recovery'] = (full_df.recovery_unadjusted_multiplier / new_baseline_recovery_unadjusted_pct).copy()

    full_df['gross_loss_impact'] = full_df['unit_loss_score'] - full_df['ms_original']
    full_df['recovery_impact'] = (full_df['unit_loss_score'] * full_df['est_unit_loss']
        * full_df['recovery_unadjusted_multiplier'] * (full_df['baselined_unadjusted_recovery'] - 1))
    full_df['ltv_impact'] = ((baseline_ltv / full_df['ltv']) - 1) * 17
    full_df['apr_impact'] = (baseline_apr - full_df['apr']) / 0.01 * 0.7
    full_df['ragu_score'] = (
        (1 - full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier) * full_df.unit_loss_score
        + full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier
        * full_df.unit_loss_score * full_df.baselined_unadjusted_recovery
        + full_df['ltv_impact']
        + full_df['apr_impact'])

    # --- STE dual-path override (MANDATORY) ---
    if ste_metrics_df is not None:
        ste_row = ste_metrics_df[ste_metrics_df.vintage == vintage]
        if len(ste_row) > 0:
            ste_row = ste_row.iloc[0]
            full_df['ms_original'] = ste_row['model_score_wtd']
            full_df['ltv'] = ste_row['ltv_wtd']
            full_df['apr'] = ste_row['apr_wtd']
            full_df['ltv_impact'] = ((baseline_ltv / full_df['ltv']) - 1) * 17
            full_df['apr_impact'] = (baseline_apr - full_df['apr']) / 0.01 * 0.7
            full_df['ragu_score'] = (
                (1 - full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier) * full_df.unit_loss_score
                + full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier
                * full_df.unit_loss_score * full_df.baselined_unadjusted_recovery
                + full_df['ltv_impact']
                + full_df['apr_impact'])

    full_df['vintage'] = vintage
    return full_df


# --- Main RAGU Loop: STE x Periods ---
all_vintages = sorted(ula_df_total['vintage'].unique())
results = []

for lob in LOBS:
    excluded = EXCLUDED_VINTAGES.get(lob, set())
    baseline_config = BASELINES[lob]

    for vintage in all_vintages:
        if vintage in excluded:
            continue
        try:
            result = get_ragu_score(
                vintage, lob, ula_df_total, new_recovery, ms_df, baseline_config,
                ste_metrics_df=ste_metrics_df
            )
            if result is not None:
                results.append(result)
        except Exception as e:
            print(f"Error: {vintage} {lob}: {e}")

    print(f"{lob} complete")

all_df = pd.concat(results, ignore_index=False)
all_df = all_df.reset_index()
print(f"\nTotal results: {len(all_df)} rows across {all_df.vintage.nunique()} vintages")
print("[PROGRESS] Scoring Complete")


STE complete

Total results: 31 rows across 31 vintages
[PROGRESS] Scoring Complete


In [10]:
# =============================================================================
# CELL 8b: ROLLUP AGGREGATION
# =============================================================================
# STE is a single LOB -- no rollups needed here.
# The control room will incorporate STE into POS/nonKMX rollups at merge time.

if ROLLUP_GROUPS:
    rollup_metrics = [
        'ms_original', 'gross_loss_impact', 'recovery_impact',
        'ltv_impact', 'apr_impact', 'ragu_score', 'ltv', 'apr',
    ]
    for group_name, group_lobs in ROLLUP_GROUPS.items():
        group_df = all_df[all_df.lob.isin(group_lobs)].copy()
        group_df = group_df.rename(columns={'amt_financed_x': 'amt_financed'})
        rollup = group_df.groupby('vintage').apply(
            weighted_average_and_sum, rollup_metrics, include_groups=False
        ).reset_index()
        rollup['lob'] = group_name
        rollup = rollup.rename(columns={'amt_financed': 'amt_financed_x'})
        all_df = pd.concat([all_df, rollup], ignore_index=True)

print(f"Final all_df: {len(all_df)} rows across {all_df.lob.nunique()} groups")
print(f"Groups: {sorted(all_df.lob.unique())}")

Final all_df: 31 rows across 1 groups
Groups: ['STE']


In [11]:
# =============================================================================
# CELL 9: UNIFIED EXCEL EXPORT
# =============================================================================

METRIC_ROWS = [
    ('Model Score',       'ms_original'),
    ('Gross Loss Impact', 'gross_loss_impact'),
    ('Recovery Impact',   'recovery_impact'),
    ('LTV Impact',        'ltv_impact'),
    ('APR Impact',        'apr_impact'),
    ('RAGU Score',        'ragu_score'),
    ('Amount Financed',   'amt_financed_x'),
    ('Weighted LTV',      'ltv'),
    ('Weighted APR',      'apr'),
]

EXCEL_SHEET_MAP = {'q': 'Data Tables (Q)', 'm': 'Data Tables (M)', 'w': 'Data Tables (W)'}
EXCEL_OUTPUT = 'ste_ragu.xlsx'

sheet_name = EXCEL_SHEET_MAP[granularity]
sorted_vintages = sorted(all_df['vintage'].unique())

if os.path.exists(EXCEL_OUTPUT):
    wb = openpyxl.load_workbook(EXCEL_OUTPUT)
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]
    ws = wb.create_sheet(sheet_name)
else:
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = sheet_name

current_row = 1
all_export_lobs = LOBS + list(ROLLUP_GROUPS.keys())

for lob in all_export_lobs:
    lob_data = all_df[all_df.lob == lob].set_index('vintage')

    ws.cell(row=current_row, column=1, value=lob)
    for col_idx, v in enumerate(sorted_vintages, start=2):
        ws.cell(row=current_row, column=col_idx, value=v)
    current_row += 1

    for label, col_key in METRIC_ROWS:
        ws.cell(row=current_row, column=1, value=label)
        for col_idx, v in enumerate(sorted_vintages, start=2):
            if v in lob_data.index:
                ws.cell(row=current_row, column=col_idx, value=lob_data.loc[v, col_key])
        current_row += 1

    current_row += 1

wb.save(EXCEL_OUTPUT)
print(f"Saved to {EXCEL_OUTPUT} (sheet: {sheet_name})")
print(f"  {len(all_export_lobs)} groups x {len(sorted_vintages)} periods")
print("[PROGRESS] Excel Export Complete")


Saved to ste_ragu.xlsx (sheet: Data Tables (W))
  1 groups x 31 periods
[PROGRESS] Excel Export Complete


In [12]:
# =============================================================================
# CELL 10: CSV OUTPUT AND OPTIONAL REDSHIFT UPLOAD
# =============================================================================

run_sandbox = False
run_historical = False

col_rename = {
    'ms_original': 'model_score',
    'gross_loss_impact': 'gross_loss',
    'recovery_impact': 'recovery',
    'ltv_impact': 'ltv',
    'apr_impact': 'apr',
}
final_cols = ['lob', 'vintage', 'model_score', 'gross_loss', 'recovery', 'ltv', 'apr', 'ragu_score', 'amt_financed_x']
all_df['month_run'] = pd.Timestamp.now().strftime('%Y M%m')

if 'ms_original' in all_df.columns:
    source_cols = ['lob', 'vintage', 'ms_original', 'gross_loss_impact', 'recovery_impact', 'ltv_impact', 'apr_impact', 'ragu_score', 'amt_financed_x']
    output_df = all_df[source_cols + ['month_run']].rename(columns=col_rename)
else:
    output_df = all_df[final_cols + ['month_run']].copy()

output_df.to_csv('ste_all_df.csv', index=False)
print(f"Saved ste_all_df.csv ({len(output_df)} rows)")

display(output_df.drop(columns='month_run').head(20))


def upload_ragu_to_redshift(df, table='sandbox.ste_ragu_monthend_current'):
    """Upload a DataFrame to a Redshift table via INSERT INTO VALUES."""
    upload_df = df.copy().reset_index(drop=True)
    upload_df['insert_column'] = (
        "('" + upload_df['lob'].astype(str)
        + "', '" + upload_df['vintage'].astype(str)
        + "', " + upload_df['model_score'].round(4).astype(str)
        + ", " + upload_df['gross_loss'].round(4).astype(str)
        + ", " + upload_df['recovery'].round(4).astype(str)
        + ", " + upload_df['ltv'].round(4).astype(str)
        + ", " + upload_df['apr'].round(4).astype(str)
        + ", " + upload_df['ragu_score'].round(4).astype(str)
        + ", '" + upload_df['month_run'].astype(str)
        + "')"
    )
    values_str = upload_df['insert_column'].str.cat(sep=',').replace("'nan'", 'null')

    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        cur = conn.cursor()
        cur.execute(f"DROP TABLE IF EXISTS {table};")
        cur.execute(f"""
            CREATE TABLE {table} (
                lob         VARCHAR(25),
                vintage     VARCHAR(20),
                model_score FLOAT,
                gross_loss  FLOAT,
                recovery    FLOAT,
                ltv         FLOAT,
                apr         FLOAT,
                ragu_score  FLOAT,
                month_run   VARCHAR(10)
            );
        """)
        cur.execute(f"INSERT INTO {table} VALUES {values_str}")
        conn.commit()
    print(f"Uploaded {len(upload_df)} rows to {table}")


if run_sandbox:
    month_run_val = output_df['month_run'].iloc[0]
    sandbox_df = output_df[output_df['vintage'] < month_run_val].drop(columns='amt_financed_x')
    print(f"Filtered to {len(sandbox_df)} rows (excluded vintages >= {month_run_val})")
    upload_ragu_to_redshift(sandbox_df)

if run_historical:
    print('Historical upload not configured for STE yet')
print("[PROGRESS] Export Complete")


Saved ste_all_df.csv (31 rows)


,lob,vintage,model_score,gross_loss,recovery,ltv,apr,ragu_score,amt_financed_x
0,STE,2025-10-05/2025-10-11,132.387991,-0.294385,2.220079,6.850531,-0.065676,139.856017,11656111.07
1,STE,2025-10-12/2025-10-18,131.324983,-0.310452,1.507937,4.379929,0.706378,136.405218,16396523.10
2,STE,2025-10-19/2025-10-25,130.586113,-0.617602,1.028487,4.868563,0.771281,137.143317,14174178.34
3,STE,2025-10-26/2025-11-01,130.742756,-0.324771,1.055220,4.413850,0.983871,137.403305,12742120.54
4,STE,2025-11-02/2025-11-08,131.189521,-0.348900,0.818265,4.627112,1.061423,137.495100,11797743.29
5,STE,2025-11-09/2025-11-15,131.151696,-0.758958,0.818535,4.365672,0.965230,137.227029,10837995.44
6,STE,2025-11-16/2025-11-22,131.413940,-0.549182,1.019251,4.047131,1.091861,137.160358,10897490.66
7,STE,2025-11-23/2025-11-29,131.409655,-0.412072,1.134020,4.390962,1.159453,138.282698,8954590.77
8,STE,2025-11-30/2025-12-06,131.885318,-0.444833,0.813894,3.971796,1.223805,137.426040,9491709.74
9,STE,2025-12-07/2025-12-13,131.511651,-0.596676,0.360353,3.989007,0.858271,136.084343,9120548.01


[PROGRESS] Export Complete


In [13]:
# =============================================================================
# CELL 11: COMPARE RAGU vs STE WEEKLY METRICS
# =============================================================================
# Diagnostic: compare ULA-derived metrics vs ste_ragu_weekly.txt to confirm
# the dual-path override is working correctly.

if ste_metrics_df is not None and len(ste_metrics_df) > 0:
    ragu_by_vintage = all_df[all_df.lob == 'STE'][['vintage', 'ms_original', 'ltv', 'apr']].copy()

    compare = ragu_by_vintage.merge(ste_metrics_df[['vintage', 'model_score_wtd', 'ltv_wtd', 'apr_wtd']],
                                     on='vintage', how='inner')

    if len(compare) > 0:
        print("=== RAGU vs STE Weekly Metrics Comparison ===")
        print(f"Total comparisons: {len(compare)}")
        print(f"\nModel Score MAE: {(compare.ms_original - compare.model_score_wtd).abs().mean():.6f}")
        print(f"LTV MAE:         {(compare.ltv - compare.ltv_wtd).abs().mean():.6f}")
        print(f"APR MAE:         {(compare.apr - compare.apr_wtd).abs().mean():.6f}")
        print("\nSample (first 10 vintages):")
        display(compare.head(10))
    else:
        print("WARNING: No overlapping vintages between RAGU output and ste_metrics_df")
else:
    print("ste_metrics_df not available; skipping comparison")

=== RAGU vs STE Weekly Metrics Comparison ===
Total comparisons: 31

Model Score MAE: 0.000000
LTV MAE:         0.000000
APR MAE:         0.000000

Sample (first 10 vintages):


,vintage,ms_original,ltv,apr,model_score_wtd,ltv_wtd,apr_wtd
0,2025-10-05/2025-10-11,132.387991,1.382778,0.250938,132.387991,1.382778,0.250938
1,2025-10-12/2025-10-18,131.324983,1.542568,0.239909,131.324983,1.542568,0.239909
2,2025-10-19/2025-10-25,130.586113,1.508101,0.238982,130.586113,1.508101,0.238982
3,2025-10-26/2025-11-01,130.742756,1.540125,0.235945,130.742756,1.540125,0.235945
4,2025-11-02/2025-11-08,131.189521,1.524938,0.234837,131.189521,1.524938,0.234837
5,2025-11-09/2025-11-15,131.151696,1.543598,0.236211,131.151696,1.543598,0.236211
6,2025-11-16/2025-11-22,131.413940,1.566959,0.234402,131.413940,1.566959,0.234402
7,2025-11-23/2025-11-29,131.409655,1.541773,0.233436,131.409655,1.541773,0.233436
8,2025-11-30/2025-12-06,131.885318,1.572588,0.232517,131.885318,1.572588,0.232517
9,2025-12-07/2025-12-13,131.511651,1.571299,0.237739,131.511651,1.571299,0.237739


In [14]:
# =============================================================================
# CELL 12: DIAGNOSTICS
# =============================================================================
# Summary statistics and sanity checks

print("=== STE RAGU Score Summary ===")
print(f"Vintages: {all_df.vintage.nunique()}")
print(f"Period range: {all_df.vintage.min()} to {all_df.vintage.max()}")
print(f"\nMean RAGU Score: {all_df.ragu_score.mean():.2f}")
print(f"Std RAGU Score:  {all_df.ragu_score.std():.2f}")
print(f"Min RAGU Score:  {all_df.ragu_score.min():.2f}")
print(f"Max RAGU Score:  {all_df.ragu_score.max():.2f}")

print("\n=== Component Averages ===")
for col in ['ms_original', 'gross_loss_impact', 'recovery_impact', 'ltv_impact', 'apr_impact']:
    if col in all_df.columns:
        print(f"  {col}: {all_df[col].mean():.4f}")

print("\n=== ULA Flag Rates ===")
flag_cols = ['high_pti_flag', 'zero_cash_down_flag', 'high_mileage_vehicle_flag',
             'car_make_penalty_flag', 'car_make_benefit_flag', 'driver_flag', 'prev_co_flag']
for flag in flag_cols:
    if flag in ula_df_total.columns:
        rate = ula_df_total[flag].mean()
        print(f"  {flag}: {rate:.4f} ({rate*100:.1f}%)")

=== STE RAGU Score Summary ===
Vintages: 31
Period range: 2025-10-05/2025-10-11 to 2026-05-03/2026-05-09

Mean RAGU Score: 139.60
Std RAGU Score:  2.70
Min RAGU Score:  136.08
Max RAGU Score:  146.53

=== Component Averages ===
  ms_original: 132.8340
  gross_loss_impact: -0.3948
  recovery_impact: 0.8095
  ltv_impact: 4.9444
  apr_impact: 1.1436

=== ULA Flag Rates ===
  high_pti_flag: 0.1542 (15.4%)
  zero_cash_down_flag: 0.1158 (11.6%)
  high_mileage_vehicle_flag: 0.0164 (1.6%)
  car_make_penalty_flag: 0.0065 (0.7%)
  car_make_benefit_flag: 0.0335 (3.4%)
  driver_flag: 0.0095 (0.9%)
  prev_co_flag: 0.0174 (1.7%)
